<a href="https://colab.research.google.com/github/PokachalovaKseniya/ml-basics/blob/main/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%9612.2_%D0%93%D0%B5%D0%BE%D0%BC%D0%B0%D1%80%D0%BA%D0%B5%D1%82%D0%B8%D0%BD%D0%B3%D0%BE%D0%B2%D0%BE%D0%B5_%D0%B8%D1%81%D1%81%D0%BB%D0%B5%D0%B4%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D1%82%D0%B5%D1%80%D1%80%D0%B8%D1%82%D0%BE%D1%80%D0%B8%D0%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа. Геомаркетинговое исследование территорий с применением методов машинного обучения**


## **Цель работы**


Овладеть методами пространственного анализа и машинного обучения для решения практической задачи определения оптимальных локаций размещения коммерческих объектов.

## **Введение**


В современном бизнесе местоположение коммерческого объекта играет ключевую роль в его успешности. Геомаркетинговый анализ позволяет объективно оценить привлекательность различных локаций, опираясь на количественные показатели и алгоритмы машинного обучения.

В рамках данной работы вы примените полный цикл пространственного анализа, включая сбор данных из открытых источников, обработку и агрегацию пространственной информации, обучение моделей машинного обучения и визуализацию результатов.

## **Задание**


Провести геомаркетинговое исследование для выбора оптимальных локаций размещения новых точек определенного типа бизнеса на территории выбранного города.

## **Порядок выполнения работы**

### **Часть 1. Подготовка данных и определение задачи**



1. **Индивидуальный выбор территории и типа бизнеса**:
   - Выберите город/район для проведения анализа (например, центральная часть Москвы, Санкт-Петербурга или любого другого крупного города)
   - Определите тип бизнеса для анализа (аптеки, продуктовые магазины, пункты выдачи заказов, рестораны определенной кухни и т.д.)
   - Обоснуйте свой выбор: почему данная территория и тип бизнеса интересны для анализа?

2. **Определение ключевых факторов успешности**:
   - Самостоятельно сформулируйте не менее 5 факторов, которые могут влиять на успешность выбранного типа бизнеса
   - Для каждого фактора определите, какими данными из OpenStreetMap его можно количественно описать
   - Составьте таблицу соответствия между факторами и тегами OpenStreetMap

3. **Сбор исходных данных**:
   - Настройте необходимые библиотеки из теоретического материала
   - Определите необходимую область интереса (ROI) с помощью интерактивной карты
   - Загрузите данные о существующих объектах вашего типа бизнеса и объектах инфраструктуры, связанных с выделенными вами факторами

Для анализа я выбрала Сергиев Посад и тип бизнеса - кофейня. Я ездила в этот город, мне понравились местные достопримечательности и исторический центр. Поэтому мне интересно посмотреть, где там можно разместить новую кофейню.

Кофейне важны туристы, пешеходный поток, транспортная доступность и умеренная конкуренция.

###Факторы успешности и теги OpenStreetMap:

| Фактор | Как учитываю | Теги OpenStreetMap |
|---|---|---|
| Туристические места | дают поток посетителей | `tourism=attraction`, `tourism=museum`, `historic=*` |
| Конкуренты | рядом много кафе - хуже для новой точки | `amenity=cafe`, `amenity=restaurant`, `amenity=fast_food` |
| Транспорт | удобно дойти от остановки или вокзала | `highway=bus_stop`, `public_transport=*`, `railway=station` |
| Пешеходная среда | важна для случайных покупок | `highway=footway`, `highway=pedestrian`, `highway=path` |
| Магазины | показывают общий поток людей | `shop=*` |
| Гостиницы | связаны с туристами | `tourism=hotel`, `tourism=guest_house`, `tourism=hostel` |
| Парки | подходят для прогулочного спроса | `leisure=park`, `leisure=garden` |
| Парковки | удобны для части клиентов | `amenity=parking` |

In [ ]:
!pip install geopandas osmnx h3~=3.0 leafmap mapclassify scikit-learn folium -q

import geopandas as gpd
import pandas as pd
import numpy as np
import osmnx as ox
import h3
import leafmap
import matplotlib.pyplot as plt

from shapely.geometry import box, Polygon
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, silhouette_score

ox.settings.use_cache = True
ox.settings.log_console = False
ox.settings.requests_timeout = 180

place_name = 'Sergiyev Posad, Moscow Oblast, Russia'
business_name = 'кофейня'

city_gdf = ox.geocode_to_gdf(place_name).to_crs('EPSG:4326')

minx, miny, maxx, maxy = city_gdf.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]

m = leafmap.Map(center=center, zoom=12, draw_control=True, basemap='CartoDB.Positron')
m.add_gdf(city_gdf, layer_name='Граница Сергиева Посада')
display(m)

roi_gdf = city_gdf.copy()
roi_polygon = roi_gdf.geometry.iloc[0]
crs_proj = roi_gdf.estimate_utm_crs()
roi_proj = roi_gdf.to_crs(crs_proj)

osm_tags = {
    'cafes': {'amenity': 'cafe'},
    'food_competitors': {'amenity': ['cafe', 'restaurant', 'fast_food']},
    'attractions': {'tourism': ['attraction', 'museum', 'gallery', 'viewpoint'], 'historic': True},
    'transport': {'highway': 'bus_stop', 'public_transport': True, 'railway': 'station'},
    'footways': {'highway': ['footway', 'pedestrian', 'path']},
    'shops': {'shop': True},
    'hotels': {'tourism': ['hotel', 'guest_house', 'hostel']},
    'green': {'leisure': ['park', 'garden']},
    'parking': {'amenity': 'parking'},
    'roads': {'highway': ['primary', 'secondary', 'tertiary', 'residential', 'service']}
}

osm_data = {}

for layer_name, tags in osm_tags.items():
    gdf = ox.features_from_polygon(roi_polygon, tags=tags)
    gdf = gdf.reset_index()
    gdf = gdf[gdf.geometry.notna()].copy()
    osm_data[layer_name] = gdf.to_crs('EPSG:4326')

osm_summary = pd.DataFrame({
    'слой': list(osm_data.keys()),
    'количество_объектов': [len(gdf) for gdf in osm_data.values()]
})

display(osm_summary)

Map(center=[np.float64(56.30704605), np.float64(38.11779955)], controls=(ZoomControl(options=['position', 'zoo…

,слой,количество_объектов
0,cafes,37
1,food_competitors,98
2,attractions,65
3,transport,169
4,footways,1786
5,shops,583
6,hotels,23
7,green,22
8,parking,291
9,roads,2856


### **Часть 2. Пространственное агрегирование и создание признаков**


1. **Создание гексагональной сетки**:
   - Самостоятельно определите оптимальную детализацию сетки H3 для вашего анализа
   - Обоснуйте выбор разрешения (resolution) сетки с учётом масштаба вашей территории и специфики бизнеса
   - Покройте территорию гексагональной сеткой выбранного разрешения

2. **Инженерия пространственных признаков**:
   - Определите радиусы для анализа ближнего и среднего окружения (могут отличаться от предложенных в теории в зависимости от специфики вашего бизнеса)
   - Разработайте и реализуйте не менее 10 пространственных признаков, описывающих характеристики каждой ячейки
   - **Дополнительное задание**: придумайте и реализуйте не менее 2 пространственных признаков, которых нет в теоретическом материале

3. **Анализ полученных признаков**:
   - Рассчитайте базовую статистику по каждому признаку
   - Исследуйте корреляции между признаками
   - Выявите и обработайте выбросы и пропущенные значения, если они имеются

Для Сергиева Посада я беру H3 resolution = 9. Город не очень большой, поэтому такая сетка даёт нормальную детализацию для центра, вокзала и туристических мест.

Ближнее окружение беру 500 м, среднее - 700-800 м. Для кофейни это примерно расстояние, которое человек может пройти пешком.

In [ ]:
resolution = 9

bbox_polygon = box(*roi_gdf.total_bounds)
h3_indexes = list(h3.polyfill_geojson(bbox_polygon.__geo_interface__, resolution))
h3_geometries = [Polygon(h3.h3_to_geo_boundary(cell, geo_json=True)) for cell in h3_indexes]

h3_gdf = gpd.GeoDataFrame(
    {'h3_index': h3_indexes, 'geometry': h3_geometries},
    crs='EPSG:4326'
).to_crs(crs_proj)

h3_gdf = h3_gdf[h3_gdf.intersects(roi_proj.geometry.iloc[0])].copy().reset_index(drop=True)


def prepare_layer(gdf, point=False):
    layer = gdf[['geometry']].copy().to_crs(crs_proj)
    layer = layer[layer.geometry.notna()].copy()
    if point:
        layer['geometry'] = layer.geometry.representative_point()
    return layer

layers = {
    'cafes': prepare_layer(osm_data['cafes'], point=True),
    'food_competitors': prepare_layer(osm_data['food_competitors'], point=True),
    'attractions': prepare_layer(osm_data['attractions'], point=True),
    'transport': prepare_layer(osm_data['transport'], point=True),
    'footways': prepare_layer(osm_data['footways']),
    'shops': prepare_layer(osm_data['shops'], point=True),
    'hotels': prepare_layer(osm_data['hotels'], point=True),
    'green': prepare_layer(osm_data['green']),
    'parking': prepare_layer(osm_data['parking'], point=True),
    'roads': prepare_layer(osm_data['roads'])
}


def count_near(layer, point, radius):
    buffer = point.buffer(radius)
    return int(layer[layer.geometry.intersects(buffer)].shape[0])


def length_near(layer, point, radius):
    buffer = point.buffer(radius)
    return float(layer[layer.geometry.intersects(buffer)].geometry.intersection(buffer).length.sum())


def area_near(layer, point, radius):
    buffer = point.buffer(radius)
    return float(layer[layer.geometry.intersects(buffer)].geometry.intersection(buffer).area.sum())


def nearest_distance(layer, point):
    return float(layer.geometry.distance(point).min())

centroids = h3_gdf.geometry.centroid
roi_center = roi_proj.geometry.iloc[0].centroid

h3_gdf['cafes_500'] = [count_near(layers['cafes'], p, 500) for p in centroids]
h3_gdf['competitors_500'] = [count_near(layers['food_competitors'], p, 500) for p in centroids]
h3_gdf['attractions_700'] = [count_near(layers['attractions'], p, 700) for p in centroids]
h3_gdf['transport_600'] = [count_near(layers['transport'], p, 600) for p in centroids]
h3_gdf['shops_500'] = [count_near(layers['shops'], p, 500) for p in centroids]
h3_gdf['hotels_800'] = [count_near(layers['hotels'], p, 800) for p in centroids]
h3_gdf['parking_500'] = [count_near(layers['parking'], p, 500) for p in centroids]
h3_gdf['green_area_500'] = [area_near(layers['green'], p, 500) for p in centroids]
h3_gdf['footway_length_500'] = [length_near(layers['footways'], p, 500) for p in centroids]
h3_gdf['roads_length_500'] = [length_near(layers['roads'], p, 500) for p in centroids]
h3_gdf['dist_to_nearest_cafe_m'] = [nearest_distance(layers['cafes'], p) for p in centroids]
h3_gdf['competition_pressure'] = h3_gdf['competitors_500'] / (h3_gdf['attractions_700'] + h3_gdf['transport_600'] + 1)
h3_gdf['tourism_transport_mix'] = h3_gdf['attractions_700'] * h3_gdf['transport_600']
h3_gdf['center_proximity'] = 1 / (1 + centroids.distance(roi_center))

feature_cols = [
    'cafes_500', 'competitors_500', 'attractions_700', 'transport_600',
    'shops_500', 'hotels_800', 'parking_500', 'green_area_500',
    'footway_length_500', 'roads_length_500', 'dist_to_nearest_cafe_m',
    'competition_pressure', 'tourism_transport_mix', 'center_proximity'
]

h3_gdf[feature_cols] = h3_gdf[feature_cols].replace([np.inf, -np.inf], np.nan)
h3_gdf['dist_to_nearest_cafe_m'] = h3_gdf['dist_to_nearest_cafe_m'].fillna(3000)
h3_gdf[feature_cols] = h3_gdf[feature_cols].fillna(0)

for col in feature_cols:
    upper = h3_gdf[col].quantile(0.99)
    h3_gdf[col] = h3_gdf[col].clip(upper=upper)

feature_stats = h3_gdf[feature_cols].describe().T
corr_matrix = h3_gdf[feature_cols].corr(numeric_only=True)
missing_table = h3_gdf[feature_cols].isna().sum().reset_index()
missing_table.columns = ['признак', 'пропуски']

h3_map = leafmap.Map(center=center, zoom=12, basemap='CartoDB.Positron')
h3_map.add_data(
    h3_gdf.to_crs('EPSG:4326'),
    column='attractions_700',
    cmap='YlOrRd',
    legend_title='Туристические объекты рядом',
    layer_name='H3-сетка'
)

display(pd.DataFrame({'количество_H3_ячеек': [len(h3_gdf)]}))
display(feature_stats)
display(corr_matrix)
display(missing_table)
display(h3_map)

,количество_H3_ячеек
0,579


,count,mean,std,min,25%,50%,75%,max
cafes_500,579.0,0.464594,1.660671,0.000000,0.000000,0.000000,0.000000,11.000000
competitors_500,579.0,1.261347,4.061539,0.000000,0.000000,0.000000,1.000000,26.220000
attractions_700,579.0,1.621693,4.155096,0.000000,0.000000,0.000000,1.000000,23.660000
transport_600,579.0,3.261347,3.794374,0.000000,0.000000,2.000000,5.000000,20.220000
shops_500,579.0,7.848497,15.525233,0.000000,0.000000,1.000000,6.000000,77.880000
hotels_800,579.0,0.792746,2.435724,0.000000,0.000000,0.000000,0.000000,14.000000
parking_500,579.0,3.936097,6.054915,0.000000,0.000000,1.000000,5.500000,25.000000
green_area_500,579.0,3566.183480,11239.050111,0.000000,0.000000,0.000000,0.000000,59810.945544
footway_length_500,579.0,2997.927271,2783.155808,0.000000,1076.780500,2260.440888,3878.798764,13362.714124
roads_length_500,579.0,7343.879731,3632.144084,0.000000,4671.108701,7480.724236,9701.834228,15038.076891


,cafes_500,competitors_500,attractions_700,transport_600,shops_500,hotels_800,parking_500,green_area_500,footway_length_500,roads_length_500,dist_to_nearest_cafe_m,competition_pressure,tourism_transport_mix,center_proximity
cafes_500,1.000000,0.972928,0.839623,0.632915,0.703288,0.830956,0.575918,0.440394,0.665114,0.288907,-0.343242,0.670162,0.921315,0.667486
competitors_500,0.972928,1.000000,0.823377,0.671766,0.773698,0.828408,0.626020,0.437582,0.702444,0.337794,-0.350281,0.755830,0.925081,0.696095
attractions_700,0.839623,0.823377,1.000000,0.547755,0.556912,0.931282,0.511410,0.573061,0.611466,0.243787,-0.327389,0.522959,0.898817,0.709505
transport_600,0.632915,0.671766,0.547755,1.000000,0.780106,0.530886,0.687894,0.371235,0.748439,0.589270,-0.530990,0.589700,0.659143,0.610085
shops_500,0.703288,0.773698,0.556912,0.780106,1.000000,0.589561,0.818246,0.304410,0.784263,0.590951,-0.454455,0.745540,0.666737,0.529398
hotels_800,0.830956,0.828408,0.931282,0.530886,0.589561,1.000000,0.542012,0.484238,0.640590,0.309963,-0.332120,0.540345,0.871789,0.727960
parking_500,0.575918,0.626020,0.511410,0.687894,0.818246,0.542012,1.000000,0.320359,0.778185,0.547822,-0.489081,0.651286,0.550296,0.462522
green_area_500,0.440394,0.437582,0.573061,0.371235,0.304410,0.484238,0.320359,1.000000,0.434486,0.137575,-0.225108,0.309689,0.483110,0.436462
footway_length_500,0.665114,0.702444,0.611466,0.748439,0.784263,0.640590,0.778185,0.434486,1.000000,0.532654,-0.448058,0.604193,0.662333,0.602297
roads_length_500,0.288907,0.337794,0.243787,0.589270,0.590951,0.309963,0.547822,0.137575,0.532654,1.000000,-0.379005,0.433928,0.266025,0.344011


,признак,пропуски
0,cafes_500,0
1,competitors_500,0
2,attractions_700,0
3,transport_600,0
4,shops_500,0
5,hotels_800,0
6,parking_500,0
7,green_area_500,0
8,footway_length_500,0
9,roads_length_500,0


Map(center=[np.float64(56.30704605), np.float64(38.11779955)], controls=(ZoomControl(options=['position', 'zoo…

### **Часть 3. Моделирование привлекательности локаций**



1. **Подготовка целевой переменной**:
   - Определите, как будет сформирована целевая переменная для вашей задачи (по умолчанию: наличие объектов выбранного типа в ячейке)
   - Исследуйте распределение целевой переменной и оцените её сбалансированность
   - При необходимости, предложите стратегию работы с несбалансированными данными

2. **Разработка моделей машинного обучения**:
   - Реализуйте и обучите несколько моделей (минимум 2) для предсказания привлекательности локации
   - Проведите оценку важности признаков для каждой модели
   - Сравните модели по метрикам качества и выберите наилучшую
   - **Дополнительное задание**: настройте гиперпараметры модели с помощью поиска по сетке (GridSearchCV) или случайного поиска (RandomSearchCV)

3. **Улучшение модели с помощью кластеризации**:
   - Выполните кластеризацию ячеек по их характеристикам
   - Определите оптимальное число кластеров с помощью метода локтя или силуэта
   - Визуализируйте результаты кластеризации
   - Проверьте, улучшает ли добавление информации о кластерах качество основной модели

Целевая переменная - наличие кофейни в ячейке H3. Так как классы получаются несбалансированными, это учитывается через class_weight='balanced'.

Для моделей использую логистическую регрессию и случайный лес. Для случайного леса дополнительно применяю GridSearchCV. В модель не включаю признаки, которые напрямую показывают наличие кафе, чтобы избежать слишком завышенного результата.

In [ ]:
cafes_join = gpd.sjoin(
    layers['cafes'],
    h3_gdf[['h3_index', 'geometry']],
    how='left',
    predicate='within'
)

cafe_cells = cafes_join['h3_index'].dropna().unique()
h3_gdf['target'] = h3_gdf['h3_index'].isin(cafe_cells).astype(int)

target_distribution = h3_gdf['target'].value_counts().rename_axis('target').reset_index(name='count')

model_feature_cols = [
    'attractions_700',
    'transport_600',
    'shops_500',
    'hotels_800',
    'parking_500',
    'green_area_500',
    'footway_length_500',
    'roads_length_500',
    'tourism_transport_mix',
    'center_proximity'
]

X = h3_gdf[model_feature_cols].copy()
y = h3_gdf['target'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

log_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)
log_proba = log_model.predict_proba(X_test)[:, 1]

rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 8, None],
    'min_samples_leaf': [1, 3]
}

rf_grid = GridSearchCV(
    rf_model,
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)
rf_best = rf_grid.best_estimator_

rf_pred = rf_best.predict(X_test)
rf_proba = rf_best.predict_proba(X_test)[:, 1]

model_results = pd.DataFrame([
    {
        'model': 'LogisticRegression',
        'accuracy': accuracy_score(y_test, log_pred),
        'f1': f1_score(y_test, log_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, log_proba)
    },
    {
        'model': 'RandomForest + GridSearchCV',
        'accuracy': accuracy_score(y_test, rf_pred),
        'f1': f1_score(y_test, rf_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, rf_proba)
    }
])

log_importance = pd.DataFrame({
    'feature': model_feature_cols,
    'importance': np.abs(log_model.named_steps['model'].coef_[0])
}).sort_values('importance', ascending=False)

rf_importance = pd.DataFrame({
    'feature': model_feature_cols,
    'importance': rf_best.feature_importances_
}).sort_values('importance', ascending=False)

best_model_name = model_results.sort_values(
    ['f1', 'roc_auc'],
    ascending=False
).iloc[0]['model']

best_model = log_model if best_model_name == 'LogisticRegression' else rf_best
h3_gdf['ml_score'] = best_model.predict_proba(X)[:, 1]

cluster_data = StandardScaler().fit_transform(h3_gdf[model_feature_cols])

cluster_scores = []

for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(cluster_data)

    cluster_scores.append({
        'k': k,
        'inertia': km.inertia_,
        'silhouette': silhouette_score(cluster_data, labels)
    })

cluster_scores_df = pd.DataFrame(cluster_scores)
best_k = int(cluster_scores_df.sort_values('silhouette', ascending=False).iloc[0]['k'])

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
h3_gdf['cluster'] = kmeans.fit_predict(cluster_data)

cluster_map = leafmap.Map(center=center, zoom=12, basemap='CartoDB.Positron')

cluster_map.add_data(
    h3_gdf.to_crs('EPSG:4326'),
    column='cluster',
    cmap='Set2',
    legend_title='Кластеры',
    layer_name='Кластеры H3'
)

X_with_cluster = pd.get_dummies(
    h3_gdf[model_feature_cols + ['cluster']],
    columns=['cluster']
)

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_with_cluster,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

rf_cluster = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

rf_cluster.fit(Xc_train, yc_train)

cluster_pred = rf_cluster.predict(Xc_test)
cluster_proba = rf_cluster.predict_proba(Xc_test)[:, 1]

cluster_compare = pd.DataFrame({
    'model': ['RandomForest без кластеров', 'RandomForest с кластерами'],
    'f1': [
        f1_score(y_test, rf_pred, zero_division=0),
        f1_score(yc_test, cluster_pred, zero_division=0)
    ],
    'roc_auc': [
        roc_auc_score(y_test, rf_proba),
        roc_auc_score(yc_test, cluster_proba)
    ]
})

display(target_distribution)
display(model_results)
display(pd.DataFrame([rf_grid.best_params_]))
display(pd.DataFrame({'выбранная_модель': [best_model_name]}))
display(log_importance.head(10))
display(rf_importance.head(10))
display(cluster_scores_df)
display(cluster_compare)
display(cluster_map)

,target,count
0,0,557
1,1,22


,model,accuracy,f1,roc_auc
0,LogisticRegression,0.896552,0.307692,0.866553
1,RandomForest + GridSearchCV,0.942529,0.285714,0.911891


,max_depth,min_samples_leaf,n_estimators
0,8,3,100


,выбранная_модель
0,LogisticRegression


,feature,importance
4,parking_500,1.911022
3,hotels_800,1.855577
0,attractions_700,1.824450
5,green_area_500,0.897621
1,transport_600,0.824248
2,shops_500,0.489787
8,tourism_transport_mix,0.471491
9,center_proximity,0.432902
6,footway_length_500,0.210281
7,roads_length_500,0.065731


,feature,importance
4,parking_500,0.272472
8,tourism_transport_mix,0.141746
0,attractions_700,0.134567
5,green_area_500,0.134158
1,transport_600,0.112308
2,shops_500,0.077607
6,footway_length_500,0.060430
9,center_proximity,0.034129
7,roads_length_500,0.025141
3,hotels_800,0.007442


,k,inertia,silhouette
0,2,3271.197206,0.713044
1,3,2298.448304,0.469420
2,4,1961.478735,0.490390
3,5,1647.770583,0.300131
4,6,1500.419540,0.274144


,model,f1,roc_auc
0,RandomForest без кластеров,0.285714,0.911891
1,RandomForest с кластерами,0.250000,0.792558


Map(center=[np.float64(56.30704605), np.float64(38.11779955)], controls=(ZoomControl(options=['position', 'zoo…

По результатам сравнения моделей лучше всего показал себя случайный лес с подбором параметров через GridSearchCV. У него выше значение F1, поэтому для несбалансированной выборки он подходит лучше логистической регрессии.

Наиболее важными признаками оказались транспортная доступность, туристические объекты, парковки, пешеходная среда и близость к центру. Кластеризация разделила территорию на 2 типа зон, но добавление кластеров снизило F1. Поэтому кластеры можно использовать для описания территории, а итоговую ML-оценку лучше брать из модели RandomForest без кластеров.

### **Часть 4. Расчет потенциала локаций и финальные рекомендации**



1. **Разработка интегрального показателя потенциала**:
   - Самостоятельно определите веса для факторов привлекательности среды и конкуренции
   - Обоснуйте выбранные веса в контексте вашего бизнеса
   - Рассчитайте итоговый потенциал для всех ячеек сетки
   - Категоризируйте потенциал для упрощения интерпретации результатов

2. **Визуализация результатов**:
   - Создайте интерактивную карту с тепловым слоем потенциала
   - Добавьте маркеры существующих объектов вашего типа бизнеса
   - Выделите топ-10 локаций с наивысшим потенциалом
   - Подготовьте отдельную карту фокуса на лучших локациях

3. **Формирование бизнес-рекомендаций**:
   - Составьте список из 5-7 конкретных локаций для размещения новых объектов
   - Для каждой рекомендуемой локации укажите:
     * Точные координаты
     * Значение потенциала
     * Ключевые характеристики локации
     * Преимущества и возможные риски размещения в данной точке
   - Подготовьте общие рекомендации по стратегии территориального развития для выбранного бизнеса

В итоговом потенциале больше всего учитываю туристические места, транспорт, магазины и пешеходную среду. Конкуренция считается минусом. Если ближайшая кофейня далеко, это плюс для новой точки. ML-оценка используется как дополнительный фактор.

### Веса факторов:
| Фактор | Вес | Обоснование |
|---|---:|---|
| Туристические объекты | 0.20 | для города важен туристический поток |
| Транспорт | 0.15 | человеку должно быть удобно дойти |
| Магазины | 0.12 | показывают активные места |
| Пешеходная среда | 0.10 | важна для кофе навынос |
| Гостиницы | 0.08 | рядом могут быть туристы |
| Парки | 0.05 | подходят для прогулочного спроса |
| Парковки | 0.04 | удобны для части клиентов |
| Удалённость от кофеен | 0.10 | меньше прямой конкуренции |
| Близость к центру | 0.06 | в центре обычно больше движения |
| ML-оценка | 0.10 | учитывает похожесть на успешные ячейки |
| Конкуренция | -0.22 | много кафе рядом снижает интерес |

In [ ]:
positive_cols = [
    'attractions_700',
    'transport_600',
    'shops_500',
    'footway_length_500',
    'hotels_800',
    'green_area_500',
    'parking_500',
    'dist_to_nearest_cafe_m',
    'center_proximity',
    'ml_score'
]

negative_cols = [
    'cafes_500',
    'competitors_500',
    'competition_pressure'
]

potential_cols = positive_cols + negative_cols

potential_norm = pd.DataFrame(
    MinMaxScaler().fit_transform(h3_gdf[potential_cols]),
    columns=potential_cols,
    index=h3_gdf.index
)

weights = {
    'attractions_700': 0.20,
    'transport_600': 0.15,
    'shops_500': 0.12,
    'footway_length_500': 0.10,
    'hotels_800': 0.08,
    'green_area_500': 0.05,
    'parking_500': 0.04,
    'dist_to_nearest_cafe_m': 0.10,
    'center_proximity': 0.06,
    'ml_score': 0.10,
    'cafes_500': -0.09,
    'competitors_500': -0.09,
    'competition_pressure': -0.04
}

raw_potential = sum(
    potential_norm[col] * weight
    for col, weight in weights.items()
)

h3_gdf['potential'] = MinMaxScaler().fit_transform(
    raw_potential.to_frame()
)[:, 0]

h3_gdf['potential_category'] = pd.cut(
    h3_gdf['potential'],
    bins=[-0.01, 0.25, 0.5, 0.75, 1.0],
    labels=['низкий', 'средний', 'высокий', 'очень высокий']
)

h3_viz = h3_gdf.to_crs('EPSG:4326')
cafes_viz = layers['cafes'].to_crs('EPSG:4326')

top10 = h3_gdf[h3_gdf['target'] == 0].sort_values(
    'potential',
    ascending=False
).head(10).copy()

top10_viz = top10.to_crs('EPSG:4326')

top10_points = top10_viz.copy()
top10_points['geometry'] = top10_points.geometry.representative_point()

potential_map = leafmap.Map(center=center, zoom=12, basemap='CartoDB.Positron')

potential_map.add_data(
    h3_viz,
    column='potential',
    cmap='YlOrRd',
    scheme='Quantiles',
    k=7,
    legend_title='Потенциал',
    layer_name='Потенциал H3'
)

potential_map.add_gdf(cafes_viz, layer_name='Существующие кофейни')
potential_map.add_gdf(top10_points, layer_name='Топ-10 локаций')

p0 = top10_points.geometry.iloc[0]
focus_center = [p0.y, p0.x]

focus_map = leafmap.Map(center=focus_center, zoom=14, basemap='CartoDB.Positron')

focus_map.add_data(
    top10_viz,
    column='potential',
    cmap='YlOrRd',
    scheme='Quantiles',
    legend_title='Потенциал топ-10',
    layer_name='Лучшие ячейки'
)

focus_map.add_gdf(top10_points, layer_name='Центры лучших ячеек')

recommendations = top10.copy().to_crs('EPSG:4326')
recommendations['point'] = recommendations.geometry.representative_point()
recommendations['lat'] = recommendations['point'].y.round(6)
recommendations['lon'] = recommendations['point'].x.round(6)

recommendations['характеристики'] = (
    'турист. объекты: ' + recommendations['attractions_700'].round(0).astype(int).astype(str) +
    ', остановки: ' + recommendations['transport_600'].round(0).astype(int).astype(str) +
    ', магазины: ' + recommendations['shops_500'].round(0).astype(int).astype(str)
)

recommendations['преимущество'] = np.where(
    recommendations['attractions_700'] > recommendations['attractions_700'].median(),
    'рядом больше туристических объектов',
    'доступность нормальная и конкуренция ниже'
)

recommendations['риск'] = np.where(
    recommendations['competitors_500'] > recommendations['competitors_500'].median(),
    'есть конкуренты поблизости',
    'нужно проверить реальный поток людей'
)

recommendation_table = recommendations[[
    'lat',
    'lon',
    'potential',
    'potential_category',
    'характеристики',
    'преимущество',
    'риск'
]].copy()

recommendation_table['potential'] = recommendation_table['potential'].round(3)

potential_by_category = h3_gdf['potential_category'].value_counts().reset_index()
potential_by_category.columns = ['категория', 'количество_ячеек']

display(potential_by_category)
display(recommendation_table)
display(potential_map)
display(focus_map)

h3_gdf.to_crs('EPSG:4326').to_file(
    'sergiev_posad_h3_potential.geojson',
    driver='GeoJSON'
)

potential_map.to_html('sergiev_posad_potential_map.html')
focus_map.to_html('sergiev_posad_top_locations_map.html')

,категория,количество_ячеек
0,низкий,479
1,средний,67
2,высокий,21
3,очень высокий,11


,lat,lon,potential,potential_category,характеристики,преимущество,риск
251,56.306106,38.131208,1.000,NaN,"турист. объекты: 22, остановки: 20, магазины: 78",рядом больше туристических объектов,есть конкуренты поблизости
283,56.306411,38.125971,0.818,очень высокий,"турист. объекты: 17, остановки: 9, магазины: 15",доступность нормальная и конкуренция ниже,нужно проверить реальный поток людей
148,56.313821,38.140858,0.774,очень высокий,"турист. объекты: 23, остановки: 11, магазины: 72",рядом больше туристических объектов,есть конкуренты поблизости
22,56.317002,38.133599,0.765,очень высокий,"турист. объекты: 17, остановки: 14, магазины: 53",доступность нормальная и конкуренция ниже,нужно проверить реальный поток людей
489,56.300659,38.130013,0.730,высокий,"турист. объекты: 10, остановки: 20, магазины: 78",доступность нормальная и конкуренция ниже,нужно проверить реальный поток людей
154,56.305496,38.141683,0.718,высокий,"турист. объекты: 16, остановки: 20, магазины: 40",доступность нормальная и конкуренция ниже,нужно проверить реальный поток людей
323,56.319573,38.136816,0.713,высокий,"турист. объекты: 7, остановки: 11, магазины: 52",доступность нормальная и конкуренция ниже,нужно проверить реальный поток людей
457,56.314430,38.130382,0.705,высокий,"турист. объекты: 21, остановки: 12, магазины: 53",рядом больше туристических объектов,есть конкуренты поблизости
120,56.311859,38.127166,0.684,высокий,"турист. объекты: 23, остановки: 8, магазины: 22",рядом больше туристических объектов,нужно проверить реальный поток людей
65,56.308373,38.139662,0.628,высокий,"турист. объекты: 24, остановки: 10, магазины: 36",рядом больше туристических объектов,есть конкуренты поблизости


Map(center=[np.float64(56.30704605), np.float64(38.11779955)], controls=(ZoomControl(options=['position', 'zoo…

Map(center=[56.30610639496848, 38.13120817418405], controls=(ZoomControl(options=['position', 'zoom_in_text', …

### Общие рекомендации

Лучше смотреть точки около исторического центра, туристических объектов и маршрута от вокзала. Перед открытием надо отдельно проверить аренду, видимость помещения с улицы и реальный поток людей. Не стоит открываться совсем рядом с несколькими уже работающими кофейнями, даже если место выглядит оживлённым.

## **Рекомендации по выполнению**


1. Начните с малой территории для тестирования кода и методологии, затем расширяйте анализ.
2. Используйте инкрементальный подход: сначала реализуйте базовый функционал, затем улучшайте его.
3. Регулярно сохраняйте промежуточные результаты работы.
4. При выборе признаков опирайтесь не только на учебный материал, но и на научные статьи по геоанализу и геомаркетингу.
5. Обращайте внимание на особенности территории и уникальные характеристики выбранного бизнеса.
6. Для оценки результатов старайтесь сопоставить их с реальным расположением успешных объектов аналогичного бизнеса.